# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, identifying their `@id`s and the fields and columns they contain.

We'll print the available record sets in the dataset and show, for each record set, the fields (`@id` values), field names, and corresponding data types.

In [ ]:
# Print all record sets and their fields with @ids and types

print('Available RecordSets:')
for rs in metadata.record_sets:
    print(f"  @id: {rs.id} | Name: {rs.name or '<unnamed>'}")
    print('    Fields:')
    for field in rs.fields:
        print(f"      - @id: {field.id} | Name: {field.name} | DataType: {field.data_type}")
    print('    Columns:')
    for col in getattr(rs, 'columns', []):
        print(f"      - @id: {col.id} | Name: {col.name} | DataType: {col.data_type}")
    print()

## 3. Data Extraction

Load data from each available record set into a DataFrame for analysis. Use record set and field `@id`s as displayed above.

In [ ]:
# List all available record set ids
record_sets_ids = [rs.id for rs in metadata.record_sets]
print('Record set @ids found:', record_sets_ids)

# Load all records into pandas DataFrames
dataframes = {}
for record_set_id in record_sets_ids:
    # Show user what is being loaded
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}")
    print()

# Example: show first rows from the first RecordSet
if len(record_sets_ids) > 0:
    first_record_set = record_sets_ids[0]
    print(f"First 5 records for RecordSet @id: {first_record_set}")
    display(dataframes[first_record_set].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Example: operating on the first record set
record_set_id = record_sets_ids[0] if record_sets_ids else None
df = dataframes[record_set_id]

# Show numeric columns (fields/columns), using their @id!
numeric_columns = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric columns found:", numeric_columns)

if numeric_columns:
    # Pick first numeric field as the numeric_field_id
    numeric_field_id = numeric_columns[0]
    threshold = df[numeric_field_id].quantile(0.75)  # e.g., top 25%
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (upper quartile):")
    display(filtered_df.head())

    # Normalized column
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to pick a category column for grouping
    category_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < len(df) // 2]
    group_field = category_fields[0] if category_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print('No numeric columns found for EDA.')

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. We'll produce a histogram of the selected numeric field and, if a categorical group is available, a boxplot/group plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric columns to visualize.")

## 6. Conclusion

In this notebook, we've loaded the FAIR² mass spectrometry protein dataset, explored its schema, loaded all available records by RecordSet `@id` with `mlcroissant`, and performed basic EDA and visualization. Use the `@id` fields to ensure unambiguous reference to all dataset entities. Further analysis can include feature engineering, statistical tests, or modeling depending on the research question.